In [5]:
pip install pandas numpy scikit-learn nltk

In [6]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re

In [12]:
# Download required NLTK data
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
# Initialize lemmatizer and stop words
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [9]:
def preprocess_text(text):
    """Preprocess text by tokenizing, removing stop words, and lemmatizing."""
    # Convert to lowercase and tokenize
    tokens = word_tokenize(text.lower())

    # Remove special characters and numbers
    tokens = [re.sub(r'[^a-zA-Z]', '', token) for token in tokens]
    tokens = [token for token in tokens if token]

    # Remove stop words and lemmatize
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words]

    return ' '.join(tokens)

In [10]:
# Load the dataset
df = pd.read_csv("/content/recipes_combined_dataset.csv")

In [13]:
# Preprocess ingredients
df['processed_ingredients'] = df['ingredients'].apply(preprocess_text)

In [15]:
# Create TF-IDF vectors
tfidf = TfidfVectorizer(max_features=4000)
tfidf_matrix = tfidf.fit_transform(df['processed_ingredients'])

In [16]:
# Create a DataFrame with recipe names and their TF-IDF vectors
result_df = pd.DataFrame({
    'recipeNames': df['recipeNames'],
    'features': [row for row in tfidf_matrix.toarray()]
})

In [17]:
# Display first few rows
print("Recipe Features:")
print(result_df[['recipeNames']].head(20))
print("\nFeature matrix shape:", tfidf_matrix.shape)

Recipe Features:
                            recipeNames
0           Poppy Seed Bread with Glaze
1                 Czech Christmas Hoska
2                    Applesauce Bread I
3                    Raisin Brown Bread
4               Applesauce Raisin Bread
5                    Apple Raisin Bread
6              Buttermilk Oatmeal Bread
7                           Kolaches II
8                  Whole Wheat Bread II
9                         Finnish Pulla
10                 Chinese Steamed Buns
11              Cottage Cheese Bread II
12  Mocha Chocolate Chip Banana Muffins
13             Ruby's Special Cornbread
14                        Maple Muffins
15                       French Toast I
16                Snow Day French Toast
17                    Angel Biscuits II
18         Fudgy Chocolate Chip Muffins
19                Pumpernickel Bread II

Feature matrix shape: (9999, 1330)


In [18]:
# Calculate cosine similarities
cosine_sim = cosine_similarity(tfidf_matrix)

In [19]:
# Calculate centered cosine similarities
def centered_cosine_similarity(matrix):
    centered_matrix = matrix - np.mean(matrix, axis=1, keepdims=True)
    return cosine_similarity(centered_matrix)

centered_cosine_sim = centered_cosine_similarity(tfidf_matrix.toarray())

In [20]:
# Display first 10 cosine similarities
print("\nCosine Similarities:")
count = 0
for i in range(len(df)):
    for j in range(i + 1, len(df)):
        similarity = cosine_sim[i][j]
        print(f"Cosine similarity between recipe {i+1} ({df['recipeNames'][i]}) "
              f"and recipe {j+1} ({df['recipeNames'][j]}): {similarity:.8f}")
        count += 1
        if count >= 10:
            break
    if count >= 10:
        break


Cosine Similarities:
Cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 2 (Czech Christmas Hoska): 0.10379908
Cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 3 (Applesauce Bread I): 0.39209942
Cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 4 (Raisin Brown Bread): 0.19700357
Cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 5 (Applesauce Raisin Bread): 0.36181663
Cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 6 (Apple Raisin Bread): 0.43214689
Cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 7 (Buttermilk Oatmeal Bread): 0.41706326
Cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 8 (Kolaches II): 0.24194759
Cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 9 (Whole Wheat Bread II): 0.18820491
Cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 10 (F

In [21]:
# Display first 10 centered cosine similarities
print("\nCentered Cosine Similarities:")
count = 0
for i in range(len(df)):
    for j in range(i + 1, len(df)):
        similarity = centered_cosine_sim[i][j]
        print(f"Centered cosine similarity between recipe {i+1} ({df['recipeNames'][i]}) "
              f"and recipe {j+1} ({df['recipeNames'][j]}): {similarity:.8f}")
        count += 1
        if count >= 10:
            break
    if count >= 10:
        break


Centered Cosine Similarities:
Centered cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 2 (Czech Christmas Hoska): 0.09563169
Centered cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 3 (Applesauce Bread I): 0.38483068
Centered cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 4 (Raisin Brown Bread): 0.18769483
Centered cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 5 (Applesauce Raisin Bread): 0.35391560
Centered cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 6 (Apple Raisin Bread): 0.42461901
Centered cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 7 (Buttermilk Oatmeal Bread): 0.41035882
Centered cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 8 (Kolaches II): 0.23575415
Centered cosine similarity between recipe 1 (Poppy Seed Bread with Glaze) and recipe 9 (Whole Wheat Bread II): 0.17792808


In [22]:
# Function to get top N similar recipes
def get_similar_recipes(recipe_index, n=5, centered=False):
    """Get top N similar recipes for a given recipe index."""
    sim_scores = centered_cosine_sim[recipe_index] if centered else cosine_sim[recipe_index]
    sim_scores = list(enumerate(sim_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]  # Exclude self
    recipe_indices = [i[0] for i in sim_scores]
    return df['recipeNames'].iloc[recipe_indices].tolist()

In [23]:
# Example: Get top 5 similar recipes for the first recipe
print("\nTop 5 similar recipes to", df['recipeNames'][0], "(using cosine similarity):")
print(get_similar_recipes(0, 5))


Top 5 similar recipes to Poppy Seed Bread with Glaze (using cosine similarity):
['Five Flavor Pound Cake II', 'Six Flavor Glaze', 'Sugar Cookies with Buttercream Frosting', 'Lemon Poppy Seed Bread', 'Anisette Cookies I']
